In [18]:
# Imports
import h5py
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import torch
from torch.utils.data import TensorDataset, DataLoader
import random
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [ ]:
def explore_hdf5_structure(hdf5_path):
    """Recursively prints the structure of an HDF5 file."""
    print(f"\nExploring HDF5 file: {hdf5_path}")
    
    def print_item(name, obj):
        indent = "  " * (name.count('/') + 1)
        if isinstance(obj, h5py.Group):
            print(f"{indent}- Group: /{name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"{indent}- Dataset: /{name} (shape: {obj.shape}, dtype: {obj.dtype})")

    try:
        with h5py.File(hdf5_path, 'r') as f:
            f.visititems(print_item)
    except Exception as e:
        print(f"Error reading {hdf5_path}: {e}")

# Assuming your files are in the same directory as this script:
spatial_files = ["data/2018_data_spatial.hdf5", "data/2019_data_spatial.hdf5", "data/2020_data_spatial.hdf5"]

for f_name in spatial_files:
    explore_hdf5_structure(f_name)

In [ ]:
def load_pre_aggregated_data(hdf5_paths, internal_dataset_path):
    """
    Loads pre-aggregated hourly active power data from multiple HDF5 files.

    Args:
        hdf5_paths (list): A list of paths to the HDF5 files 
                           (e.g., ["data/2018_data_spatial.hdf5", ...]).
        internal_dataset_path (str): The internal HDF5 dataset path to the table 
                                     containing the aggregated data
                                     (e.g., "/NO_PV/60min/HOUSEHOLD/table").

    Returns:
        pd.DataFrame: A single DataFrame of active power (P) at hourly resolution,
                      indexed by timestamp, for all combined years.
    """
    all_yearly_dfs = []

    for hdf5_path in hdf5_paths:
        print(f"Loading data from: {hdf5_path}...")
        try:
            with h5py.File(hdf5_path, 'r') as f:
                if internal_dataset_path in f:
                    data_table = f[internal_dataset_path]
                    
                    # Extract 'index' for timestamp and 'P_TOT' for active power
                    timestamps = pd.to_datetime(data_table['index'][:], unit='s')
                    active_power = data_table['P_TOT'][:]
                    
                    df_year = pd.DataFrame({'P': active_power}, index=timestamps)
                    all_yearly_dfs.append(df_year)
                    print(f"  Loaded {len(df_year)} records for {hdf5_path}.")
                else:
                    print(f"Error: Dataset '{internal_dataset_path}' not found in {hdf5_path}. Skipping file.")
        except Exception as e:
            print(f"Error processing file {hdf5_path}: {e}")

    if all_yearly_dfs:
        # Concatenate all yearly DataFrames and sort by index (timestamp)
        final_df = pd.concat(all_yearly_dfs).sort_index()
        # Remove any duplicate timestamps if files overlap or have repeated entries
        final_df = final_df[~final_df.index.duplicated(keep='first')]
        print(f"\nSuccessfully loaded and combined data. Total unique records: {len(final_df)}")
        
        # Verify resolution is hourly. If any file wasn't perfectly hourly, 
        # this step will ensure consistency (though unlikely given the "60min" path).
        # We use .mean() or .sum() if there were actual duplicates within the same hour, 
        # but given the nature of pre-aggregated "table", a simple drop_duplicates should be fine.
        # Let's ensure strict hourly by resampling just in case, taking the sum for safety.
        final_df = final_df.resample('H').sum()
        print(f"Final data shape after hourly resampling: {final_df.shape}")
        return final_df
    else:
        print("\nNo data loaded from any files.")
        return pd.DataFrame(columns=['P'])

# --- Example Usage ---
# Paths to your HDF5 files
hdf5_files = [
    "data/2018_data_spatial.hdf5",
    "data/2019_data_spatial.hdf5",
    "data/2020_data_spatial.hdf5"
]

# The internal path to the pre-aggregated household active power table
INTERNAL_DATASET_PATH = "/NO_PV/60min/HOUSEHOLD/table"

# Load your raw, aggregated hourly data
df_raw = load_pre_aggregated_data(hdf5_files, INTERNAL_DATASET_PATH)

# Display a quick summary of the loaded data
print("\n--- Raw Aggregated Data Info ---")
print(df_raw.head())
print(df_raw.tail())
print(df_raw.info())
print(f"Min timestamp: {df_raw.index.min()}")
print(f"Max timestamp: {df_raw.index.max()}")

In [5]:
def load_and_generate_timestamps(hdf5_paths, internal_dataset_path):
    """
    Loads pre-aggregated hourly active power data from multiple HDF5 files
    and generates timestamps based on the filename year and hourly resolution.

    Args:
        hdf5_paths (list): A list of paths to the HDF5 files 
                           (e.g., ["data/2018_data_spatial.hdf5", ...]).
        internal_dataset_path (str): The internal HDF5 dataset path to the table 
                                     containing the aggregated data
                                     (e.g., "/NO_PV/60min/HOUSEHOLD/table").

    Returns:
        pd.DataFrame: A single DataFrame of active power (P) at hourly resolution,
                      indexed by timestamp, for all combined years.
    """
    all_yearly_dfs = []

    for hdf5_path in hdf5_paths:
        print(f"Loading data from: {hdf5_path}...")
        try:
            with h5py.File(hdf5_path, 'r') as f:
                if internal_dataset_path in f:
                    data_table = f[internal_dataset_path]
                    
                    # Extract the year from the filename
                    year_str = hdf5_path.split('/')[-1].split('_')[0] # Assumes format like "data/YEAR_..."
                    year = int(year_str)
                    
                    # Generate timestamps starting from Jan 1st of the year, hourly
                    start_date = datetime(year, 1, 1, 0, 0, 0)
                    
                    # The data_table's shape gives us the number of hours in the year
                    num_hours = data_table.shape[0] 
                    
                    # Create a DatetimeIndex for the year
                    generated_timestamps = pd.date_range(start=start_date, periods=num_hours, freq='H', tz='UTC')
                    
                    # Extract 'P_TOT' for active power
                    active_power = data_table['P_TOT'][:]
                    
                    df_year = pd.DataFrame({'P': active_power}, index=generated_timestamps)
                    all_yearly_dfs.append(df_year)
                    print(f"  Loaded {len(df_year)} records for {year}.")
                else:
                    print(f"Error: Dataset '{internal_dataset_path}' not found in {hdf5_path}. Skipping file.")
        except Exception as e:
            print(f"Error processing file {hdf5_path}: {e}")
            # Consider adding a more specific error for the parsing if it's not filename related
            if "invalid literal for int()" in str(e):
                 print(f"  Could not parse year from filename: {hdf5_path}. Ensure filename format is 'YEAR_data_spatial.hdf5'")

    if all_yearly_dfs:
        # Concatenate all yearly DataFrames and sort by index (timestamp)
        final_df = pd.concat(all_yearly_dfs).sort_index()
        
        # Remove any duplicate timestamps that might arise from different file boundaries (e.g. if one year's data overlaps)
        final_df = final_df[~final_df.index.duplicated(keep='first')]
        
        print(f"\nSuccessfully loaded and combined data. Total unique records: {len(final_df)}")
        
        # Verify resolution is hourly (redundant if generated correctly, but good for final check)
        # Using .mean() here if there were multiple values at the same exact timestamp after concat (unlikely with date_range)
        final_df = final_df.resample('H').sum() # Use sum to be consistent with aggregation idea
        
        print(f"Final data shape after ensuring strict hourly resolution: {final_df.shape}")
        return final_df
    else:
        print("\nNo data loaded from any files.")
        return pd.DataFrame(columns=['P'])


# --- Example Usage ---
# Paths to your HDF5 files
hdf5_files = [
    "data/2018_data_spatial.hdf5",
    "data/2019_data_spatial.hdf5",
    "data/2020_data_spatial.hdf5"
]

# The internal path to the pre-aggregated household active power table
INTERNAL_DATASET_PATH = "/NO_PV/60min/HOUSEHOLD/table"

# Load your raw, aggregated hourly data
df_raw = load_and_generate_timestamps(hdf5_files, INTERNAL_DATASET_PATH)

# Display a quick summary of the loaded data
print("\n--- Raw Aggregated Data Info ---")
print(df_raw.head())
print(df_raw.tail())
print(df_raw.info())
print(f"Min timestamp: {df_raw.index.min()}")
print(f"Max timestamp: {df_raw.index.max()}")

Loading data from: data/2018_data_spatial.hdf5...
  Loaded 8760 records for 2018.
Loading data from: data/2019_data_spatial.hdf5...
  Loaded 8760 records for 2019.
Loading data from: data/2020_data_spatial.hdf5...
  Loaded 8784 records for 2020.

Successfully loaded and combined data. Total unique records: 26304
Final data shape after ensuring strict hourly resolution: (26304, 1)

--- Raw Aggregated Data Info ---
                             P
2018-01-01 00:00:00+00:00  0.0
2018-01-01 01:00:00+00:00  0.0
2018-01-01 02:00:00+00:00  0.0
2018-01-01 03:00:00+00:00  0.0
2018-01-01 04:00:00+00:00  0.0
                                      P
2020-12-31 19:00:00+00:00  11829.777286
2020-12-31 20:00:00+00:00   9323.090116
2020-12-31 21:00:00+00:00   9972.464586
2020-12-31 22:00:00+00:00   8757.872933
2020-12-31 23:00:00+00:00   7820.059944
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26304 entries, 2018-01-01 00:00:00+00:00 to 2020-12-31 23:00:00+00:00
Freq: h
Data columns (total 1 colu

C:\Users\khurs\AppData\Local\Temp\ipykernel_1508\488514572.py:37: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  generated_timestamps = pd.date_range(start=start_date, periods=num_hours, freq='H', tz='UTC')
C:\Users\khurs\AppData\Local\Temp\ipykernel_1508\488514572.py:64: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  final_df = final_df.resample('H').sum() # Use sum to be consistent with aggregation idea


In [6]:
def split_data(df):
    train_start = '2018-05-31'
    train_end = '2019-12-31'
    val_start = '2020-01-01'
    val_end = '2020-06-30'
    test_start = '2020-07-01'
    test_end = '2020-12-31'

    # Ensure timezone awareness for comparison if df.index has timezone
    # Convert string dates to datetime objects with UTC timezone
    train_start_dt = pd.to_datetime(train_start).tz_localize('UTC')
    train_end_dt = pd.to_datetime(train_end + ' 23:00:00').tz_localize('UTC') # End of day
    val_start_dt = pd.to_datetime(val_start).tz_localize('UTC')
    val_end_dt = pd.to_datetime(val_end + ' 23:00:00').tz_localize('UTC') # End of day
    test_start_dt = pd.to_datetime(test_start).tz_localize('UTC')
    test_end_dt = pd.to_datetime(test_end + ' 23:00:00').tz_localize('UTC') # End of day

    train_df = df.loc[train_start_dt : train_end_dt]
    val_df = df.loc[val_start_dt : val_end_dt]
    test_df = df.loc[test_start_dt : test_end_dt]

    print(f"\n--- Data Split Summary ---")
    print(f"Train data range: {train_df.index.min()} to {train_df.index.max()} ({len(train_df)} hours)")
    print(f"Validation data range: {val_df.index.min()} to {val_df.index.max()} ({len(val_df)} hours)")
    print(f"Test data range: {test_df.index.min()} to {test_df.index.max()} ({len(test_df)} hours)")

    return train_df, val_df, test_df

# Run the split
train_df, val_df, test_df = split_data(df_raw)


--- Data Split Summary ---
Train data range: 2018-05-31 00:00:00+00:00 to 2019-12-31 23:00:00+00:00 (13920 hours)
Validation data range: 2020-01-01 00:00:00+00:00 to 2020-06-30 23:00:00+00:00 (4368 hours)
Test data range: 2020-07-01 00:00:00+00:00 to 2020-12-31 23:00:00+00:00 (4416 hours)


In [7]:
def normalize_data(train_df, val_df, test_df):
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # Fit on training data only to prevent data leakage
    scaler.fit(train_df[['P']])
    
    # Transform all sets using the scaler fitted on training data
    train_scaled = scaler.transform(train_df[['P']])
    val_scaled = scaler.transform(val_df[['P']])
    test_scaled = scaler.transform(test_df[['P']])

    # Convert back to DataFrame for consistent handling
    train_scaled_df = pd.DataFrame(train_scaled, index=train_df.index, columns=['P'])
    val_scaled_df = pd.DataFrame(val_scaled, index=val_df.index, columns=['P'])
    test_scaled_df = pd.DataFrame(test_scaled, index=test_df.index, columns=['P'])
    
    print(f"\n--- Normalization Summary ---")
    print(f"Train Scaled Min: {train_scaled_df['P'].min():.4f}, Max: {train_scaled_df['P'].max():.4f}")
    print(f"Val Scaled Min: {val_scaled_df['P'].min():.4f}, Max: {val_scaled_df['P'].max():.4f}")
    print(f"Test Scaled Min: {test_scaled_df['P'].min():.4f}, Max: {test_scaled_df['P'].max():.4f}")

    return train_scaled_df, val_scaled_df, test_scaled_df, scaler

# Run normalization
train_scaled_df, val_scaled_df, test_scaled_df, scaler = normalize_data(train_df, val_df, test_df)


--- Normalization Summary ---
Train Scaled Min: 0.0000, Max: 1.0000
Val Scaled Min: 0.0476, Max: 0.8754
Test Scaled Min: 0.0378, Max: 1.0001


In [8]:
def create_sequences(data, input_steps=24, output_steps=24):
    """
    Creates input-output sequences for time series forecasting.

    Args:
        data (np.ndarray): The 1D array of time series data.
        input_steps (int): Number of past steps to use as input (X).
        output_steps (int): Number of future steps to predict (Y).

    Returns:
        tuple: (X, Y) numpy arrays.
               X: (num_samples, input_steps, 1)
               Y: (num_samples, output_steps)
    """
    X, Y = [], []
    for i in range(len(data) - input_steps - output_steps + 1):
        # Input sequence: data[i] to data[i + input_steps - 1]
        X.append(data[i:(i + input_steps)])
        # Output sequence: data[i + input_steps] to data[i + input_steps + output_steps - 1]
        Y.append(data[(i + input_steps):(i + input_steps + output_steps)])
    
    X = np.array(X)
    Y = np.array(Y)
    
    # Reshape X to (num_samples, seq_len, num_features)
    # Our 'P' column is a single feature, so num_features = 1
    X = X.reshape(-1, input_steps, 1)
    
    # Y is (num_samples, output_steps) as the model output will be a 1D array of 24 predictions
    
    print(f"\n--- Sequence Creation Summary ---")
    print(f"X shape: {X.shape}")
    print(f"Y shape: {Y.shape}")

    return X, Y

# Get the 'P' values as numpy arrays for sequence creation
train_data_P = train_scaled_df['P'].values
val_data_P = val_scaled_df['P'].values
test_data_P = test_scaled_df['P'].values

# Create sequences for each split
train_X, train_Y = create_sequences(train_data_P)
val_X, val_Y = create_sequences(val_data_P)
test_X, test_Y = create_sequences(test_data_P)


--- Sequence Creation Summary ---
X shape: (13873, 24, 1)
Y shape: (13873, 24)

--- Sequence Creation Summary ---
X shape: (4321, 24, 1)
Y shape: (4321, 24)

--- Sequence Creation Summary ---
X shape: (4369, 24, 1)
Y shape: (4369, 24)


In [10]:
# Convert numpy arrays to PyTorch tensors
train_X_tensor = torch.tensor(train_X, dtype=torch.float32)
train_Y_tensor = torch.tensor(train_Y, dtype=torch.float32)
val_X_tensor = torch.tensor(val_X, dtype=torch.float32)
val_Y_tensor = torch.tensor(val_Y, dtype=torch.float32)
test_X_tensor = torch.tensor(test_X, dtype=torch.float32)
test_Y_tensor = torch.tensor(test_Y, dtype=torch.float32)

# Create TensorDatasets
# Note: .squeeze(-1) is not needed for Y if your Y shape is already (num_samples, 24)
# It would be needed if Y was (num_samples, 24, 1) and your model outputs (num_samples, 24)
train_dataset = TensorDataset(train_X_tensor, train_Y_tensor)
val_dataset = TensorDataset(val_X_tensor, val_Y_tensor)
test_dataset = TensorDataset(test_X_tensor, test_Y_tensor)

# Hyperparameters from paper (Table II)
BATCH_SIZE = 384 
LSTM_HIDDEN_SIZE = 200 

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n--- DataLoader Summary ---")
print(f"Train DataLoader batches: {len(train_loader)}")
print(f"Validation DataLoader batches: {len(val_loader)}")
print(f"Test DataLoader batches: {len(test_loader)}")


--- DataLoader Summary ---
Train DataLoader batches: 37
Validation DataLoader batches: 12
Test DataLoader batches: 12


# 1. Utility Functions

In [13]:
def set_seed(seed):
    """Sets random seeds for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # Set a random seed at the very beginning

# 2. Model Definitions

In [14]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=200, num_layers=1, output_size=24):
        super(BiLSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers, 
                              batch_first=True, bidirectional=True)
        
        # Output layer to project the BiLSTM output to the desired prediction length (24 hours)
        # Multiply hidden_size by 2 because of bidirectional (forward + backward)
        self.relu = nn.ReLU()
        self.fc_intermediate = nn.Linear(hidden_size * 2, hidden_size)
        self.fc_final = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size) e.g., (384, 24, 1)
        
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        
        # Forward propagate BiLSTM
        out, _ = self.bilstm(x, (h0, c0)) # out: (batch_size, seq_len, hidden_size * 2)
        
        # Take the output from the last time step for sequence-to-one or sequence-to-sequence if decoder isn't explicit
        # The paper's diagram implies this common approach for a direct 24-hour forecast from last state.
        out = out[:, -1, :] # Shape: (batch_size, hidden_size * 2)
        
        out = self.fc_intermediate(out)
        out = self.relu(out)
        out = self.fc_final(out) # Shape: (batch_size, output_size)
        
        return out


class CNNBiLSTMModel(nn.Module):
    def __init__(self, input_size=1, cnn_filters=64, cnn_kernel_size=3,
                 bilstm_hidden_size=200, bilstm_num_layers=1, output_size=24):
        super(CNNBiLSTMModel, self).__init__()
        self.bilstm_hidden_size = bilstm_hidden_size
        self.bilstm_num_layers = bilstm_num_layers
        self.input_size = input_size # This is number of features, usually 1

        # 1. CNN Layer: Input (batch, channels, seq_len)
        self.conv1d = nn.Conv1d(in_channels=input_size,
                                out_channels=cnn_filters,
                                kernel_size=cnn_kernel_size,
                                stride=1)
        
        # 2. Max Pooling Layer: (kernel_size=2, stride=2) halves the sequence length
        self.maxpool1d = nn.MaxPool1d(kernel_size=2, stride=2)

        # 3. Flatten Layer:
        self.flatten = nn.Flatten()

        # 4. Repeat Vector (Conceptually in PyTorch):
        self.repeat_vector_seq_len = 24 # The length of the sequence BiLSTM expects

        # Calculate input size for BiLSTM after CNN and flattening
        # Assuming input_steps = 24
        conv_output_len = 24 - cnn_kernel_size + 1 # (24 - 3 + 1) = 22
        pooled_output_len = conv_output_len // 2  # 22 // 2 = 11
        bilstm_input_features = cnn_filters * pooled_output_len # 64 * 11 = 704

        # 5. BiLSTM Layer:
        self.bilstm = nn.LSTM(bilstm_input_features, bilstm_hidden_size, bilstm_num_layers,
                              batch_first=True, bidirectional=True)

        # 6. Dense Layers:
        self.relu = nn.ReLU()
        self.fc_intermediate = nn.Linear(bilstm_hidden_size * 2, bilstm_hidden_size)
        self.fc_final = nn.Linear(bilstm_hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size) e.g., (384, 24, 1)
        
        # CNN input needs to be (batch_size, input_channels, seq_len)
        x = x.permute(0, 2, 1) # -> (batch_size, 1, 24)

        # 1. CNN
        cnn_out = self.conv1d(x) # (batch_size, cnn_filters, 22)
        
        # 2. Max Pooling
        pooled_out = self.maxpool1d(cnn_out) # (batch_size, cnn_filters, 11)
        
        # 3. Flatten
        flattened_out = self.flatten(pooled_out) # (batch_size, cnn_filters * 11 = 704)

        # 4. Repeat Vector (Conceptual in PyTorch):
        # Expand the flattened output to (batch_size, repeat_vector_seq_len, cnn_output_features)
        repeated_out = flattened_out.unsqueeze(1).repeat(1, self.repeat_vector_seq_len, 1)
        # shape: (batch_size, 24, 704)

        # 5. BiLSTM
        h0 = torch.zeros(self.bilstm_num_layers * 2, x.size(0), self.bilstm_hidden_size).to(x.device)
        c0 = torch.zeros(self.bilstm_num_layers * 2, x.size(0), self.bilstm_hidden_size).to(x.device)
        
        bilstm_out, _ = self.bilstm(repeated_out, (h0, c0))
        
        # Take the output from the last time step
        bilstm_out = bilstm_out[:, -1, :] # (batch_size, bilstm_hidden_size * 2)

        # 6. Dense Layers
        out = self.fc_intermediate(bilstm_out)
        out = self.relu(out)
        out = self.fc_final(out) # (batch_size, output_size)
        
        return out

# 3. Training and Prediction Helper Functions

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=100, patience=15):
    """Trains a PyTorch model with early stopping."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        for batch_X, batch_Y in tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False):
            batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_Y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # Validation phase
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch_X, batch_Y in tqdm(val_loader, desc=f"Epoch {epoch+1} Val", leave=False):
                batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_Y)
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)

        print(f'Epoch {epoch+1}/{num_epochs}, Train MSE: {avg_train_loss:.4f}, Val MSE: {avg_val_loss:.4f}')

        # Early stopping check (based on validation RMSE, which is sqrt of MSE)
        val_rmse = np.sqrt(avg_val_loss)
        if val_rmse < best_val_loss:
            best_val_loss = val_rmse
            epochs_no_improve = 0
            torch.save(model.state_dict(), f'best_{model.__class__.__name__}.pth') # Save best model
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f'Early stopping triggered for {model.__class__.__name__} after {patience} epochs with no improvement.')
                break
    
    print(f"Training finished for {model.__class__.__name__}. Best Val RMSE: {best_val_loss:.4f}")
    model.load_state_dict(torch.load(f'best_{model.__class__.__name__}.pth')) # Load best model for final use
    return model, history


def walk_forward_predict(model, test_df_scaled, input_steps=24, output_steps=24, scaler=None):
    """
    Performs walk-forward prediction on the test set, shifting by output_steps each time.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval() # Set model to evaluation mode
    
    all_predictions = []
    all_actuals = []

    test_data_P_scaled = test_df_scaled['P'].values

    # Iterate through the test data, taking `input_steps` input, predicting `output_steps` output
    # and then shifting the window by `output_steps` (24 hours).
    for i in tqdm(range(0, len(test_data_P_scaled) - input_steps - output_steps + 1, output_steps), desc="Walk-Forward Test"):
        
        # Get current input window (previous 24 hours) - scaled
        input_window_scaled = test_data_P_scaled[i : i + input_steps]
        
        # Get actual values for comparison (next 24 hours) - scaled
        actual_window_scaled = test_data_P_scaled[i + input_steps : i + input_steps + output_steps]
        
        # Reshape for model input: (1, seq_len, num_features)
        input_tensor = torch.tensor(input_window_scaled, dtype=torch.float32).reshape(1, input_steps, 1).to(device)
        
        with torch.no_grad():
            predicted_scaled = model(input_tensor).cpu().numpy() # Output is (1, 24)
        
        all_predictions.extend(predicted_scaled.flatten())
        all_actuals.extend(actual_window_scaled.flatten())
    
    # Invert scaling to get back to original W units
    # The scaler expects a 2D array, so we create a dummy one with the correct number of features (1)
    dummy_for_inverse_pred = np.zeros((len(all_predictions), scaler.n_features_in_))
    dummy_for_inverse_pred[:, 0] = np.array(all_predictions)
    predictions_kw = scaler.inverse_transform(dummy_for_inverse_pred)[:, 0]

    dummy_for_inverse_actual = np.zeros((len(all_actuals), scaler.n_features_in_))
    dummy_for_inverse_actual[:, 0] = np.array(all_actuals)
    actuals_kw = scaler.inverse_transform(dummy_for_inverse_actual)[:, 0]
    
    return actuals_kw, predictions_kw

# 6. Model Training and Evaluation

In [ ]:
print("\n--- Starting Model Training and Evaluation ---")

# Loss function and optimizer
criterion = nn.MSELoss()


# Train and Evaluate BiLSTM Model
print("\n===== Training BiLSTM Model =====")
bilstm_model = BiLSTMModel(hidden_size=LSTM_HIDDEN_SIZE)
optimizer_bilstm = optim.Adam(bilstm_model.parameters(), lr=0.001)
trained_bilstm, bilstm_history = train_model(bilstm_model, train_loader, val_loader, criterion, optimizer_bilstm, num_epochs=100, patience=15)

print("\n===== Evaluating BiLSTM Model on Test Set =====")
actuals_bilstm_kw, predictions_bilstm_kw = walk_forward_predict(trained_bilstm, test_scaled_df, scaler=scaler)

# Calculate final RMSE for BiLSTM
rmse_bilstm = np.sqrt(np.mean((actuals_bilstm_kw - predictions_bilstm_kw)**2))
print(f"BiLSTM Test RMSE: {rmse_bilstm:.4f} kW")


# Train and Evaluate CNN-BiLSTM Model
print("\n===== Training CNN-BiLSTM Model =====")
cnn_bilstm_model = CNNBiLSTMModel(bilstm_hidden_size=LSTM_HIDDEN_SIZE)
optimizer_cnn_bilstm = optim.Adam(cnn_bilstm_model.parameters(), lr=0.001)
trained_cnn_bilstm, cnn_bilstm_history = train_model(cnn_bilstm_model, train_loader, val_loader, criterion, optimizer_cnn_bilstm, num_epochs=100, patience=15)

print("\n===== Evaluating CNN-BiLSTM Model on Test Set =====")
actuals_cnn_bilstm_kw, predictions_cnn_bilstm_kw = walk_forward_predict(trained_cnn_bilstm, test_scaled_df, scaler=scaler)

# Calculate final RMSE for CNN-BiLSTM
rmse_cnn_bilstm = np.sqrt(np.mean((actuals_cnn_bilstm_kw - predictions_cnn_bilstm_kw)**2))
print(f"CNN-BiLSTM Test RMSE: {rmse_cnn_bilstm:.4f} kW")


# Evaluate Naive Forecasting Technique (Baseline)
print("\n===== Evaluating Naive Forecasting Technique (Baseline) =====")
naive_actuals = []
naive_predictions = []
input_steps = 24
output_steps = 24

test_data_P_kw = test_df['P'].values # Use original unscaled test data for naive
for i in range(0, len(test_data_P_kw) - input_steps - output_steps + 1, output_steps):
    # Naive prediction for the next 24 hours is simply the *preceding* 24 hours (input_window)
    # This aligns with D(t+1) = D(t) from the paper.
    naive_prediction_window_kw = test_data_P_kw[i : i + input_steps]
    actual_window_kw = test_data_P_kw[i + input_steps : i + input_steps + output_steps]
    
    if len(naive_prediction_window_kw) == output_steps: # Ensure we have a full 24-hour prediction
        naive_actuals.extend(actual_window_kw)
        naive_predictions.extend(naive_prediction_window_kw)

naive_actuals = np.array(naive_actuals)
naive_predictions = np.array(naive_predictions)

rmse_naive = np.sqrt(np.mean((naive_actuals - naive_predictions)**2))
print(f"Naive Forecasting Test RMSE: {rmse_naive:.4f} kW")


print(f"\n--- Final Results Comparison (in kW) ---")
print(f"BiLSTM Test RMSE: {rmse_bilstm / 1000:.4f} kW")
print(f"CNN-BiLSTM Test RMSE: {rmse_cnn_bilstm / 1000:.4f} kW")
print(f"Naive Forecasting Test RMSE: {rmse_naive / 1000:.4f} kW (Baseline)")


--- Starting Model Training and Evaluation ---

===== Training BiLSTM Model =====


Epoch 1/100, Train MSE: 0.0443, Val MSE: 0.0207


Epoch 2/100, Train MSE: 0.0283, Val MSE: 0.0203


Epoch 3/100, Train MSE: 0.0255, Val MSE: 0.0189


Epoch 4/100, Train MSE: 0.0207, Val MSE: 0.0152


...

Epoch 49/100, Train MSE: 0.0063, Val MSE: 0.0061


Epoch 50/100, Train MSE: 0.0063, Val MSE: 0.0062


Epoch 51/100, Train MSE: 0.0062, Val MSE: 0.0062
Early stopping triggered for CNNBiLSTMModel after 15 epochs with no improvement.
Training finished for CNNBiLSTMModel. Best Val RMSE: 0.0777

===== Evaluating CNN-BiLSTM Model on Test Set =====


Walk-Forward Test: 100%|██████████| 183/183 [00:00<00:00, 596.09it/s]

CNN-BiLSTM Test RMSE: 1467.3282 kW

===== Evaluating Naive Forecasting Technique (Baseline) =====
Naive Forecasting Test RMSE: 1786.6313 kW

--- Final Results Comparison ---
BiLSTM Test RMSE: 1482.1899 kW
CNN-BiLSTM Test RMSE: 1467.3282 kW
Naive Forecasting Test RMSE: 1786.6313 kW (Baseline)


In [ ]:
# Evaluate Naive Forecasting Technique (Baseline)
print("\n===== Evaluating Naive Forecasting Technique (Baseline) =====")
naive_actuals = []
naive_predictions = []
input_steps = 24
output_steps = 24

test_data_P_kw = test_df['P'].values # Use original unscaled test data for naive
for i in range(0, len(test_data_P_kw) - input_steps - output_steps + 1, output_steps):
    # Naive prediction for the next 24 hours is simply the *preceding* 24 hours (input_window)
    # This aligns with D(t+1) = D(t) from the paper.
    naive_prediction_window_kw = test_data_P_kw[i : i + input_steps]
    actual_window_kw = test_data_P_kw[i + input_steps : i + input_steps + output_steps]
    
    if len(naive_prediction_window_kw) == output_steps: # Ensure we have a full 24-hour prediction
        naive_actuals.extend(actual_window_kw)
        naive_predictions.extend(naive_prediction_window_kw)

naive_actuals = np.array(naive_actuals)
naive_predictions = np.array(naive_predictions)

rmse_naive = np.sqrt(np.mean((naive_actuals - naive_predictions)**2))
print(f"Naive Forecasting Test RMSE: {rmse_naive / 1000:.4f} kW")


print(f"\n--- Final Results Comparison (in kW) ---")
print(f"BiLSTM Test RMSE: {rmse_bilstm / 1000:.4f} kW")
print(f"CNN-BiLSTM Test RMSE: {rmse_cnn_bilstm / 1000:.4f} kW")
print(f"Naive Forecasting Test RMSE: {rmse_naive / 1000:.4f} kW (Baseline)")


===== Evaluating Naive Forecasting Technique (Baseline) =====
Naive Forecasting Test RMSE: 1.7866 kW

--- Final Results Comparison (in kW) ---
BiLSTM Test RMSE: 1.4822 kW
CNN-BiLSTM Test RMSE: 1.4673 kW
Naive Forecasting Test RMSE: 1.7866 kW (Baseline)
